# SuppDB Starter - Load & Explore Supplement Labels

**[SuppDB](https://supplements-nootropics-suppdb.pages.dev)** is a structured dataset of real supplement & nootropic products, built from the public **NIH DSLD** label database - one row per active ingredient, with the dose **normalized to mg**, proprietary blends flagged, and **NIH PubChem** chemistry per compound.

This free sample is **~2,250 ingredient records across ~300 real products from ~218 brands**. The full dataset (17,000+ products) is at [suppdb.net](https://supplements-nootropics-suppdb.pages.dev).

*Not medical advice - always verify against the current physical label.*

In [ ]:
import os, glob
import pandas as pd

# Works on Kaggle (attached dataset) and locally (repo samples/). On Kaggle we also
# search the input mount recursively, so it loads regardless of the dataset slug.
CANDIDATES = [
    "/kaggle/input/suppdb-supplements-sample/suppdb_sample.csv",
    "../../samples/suppdb_sample.csv",
    "../samples/suppdb_sample.csv",
    "samples/suppdb_sample.csv",
    "suppdb_sample.csv",
]
CANDIDATES += sorted(glob.glob("/kaggle/input/**/suppdb_sample.csv", recursive=True))
CANDIDATES += sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))

path = next((p for p in CANDIDATES if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError(
        "suppdb_sample.csv not found. On Kaggle, click '+ Add Input' (top-right) and "
        "add the dataset 'ahtiticheamine/suppdb-supplements-sample', then re-run this cell."
    )

df = pd.read_csv(path)
print(f"Loaded: {path}")
print(f"{len(df):,} ingredient rows | {df['product_id'].nunique():,} products | {df['brand'].nunique():,} brands")
print(f"{df.shape[1]} columns")

In [ ]:
df.head(8)

## Dosages - normalized to milligrams

Every disclosed dose is in `amount_per_serving_mg`. A `0` means the dose is hidden inside a proprietary blend (`is_proprietary_blend == 1`).

In [ ]:
# Most common ingredients in the sample
df["ingredient"].value_counts().head(12)

In [ ]:
# Typical disclosed dose (mg) for a few well-known ingredients
disclosed = df[(df["amount_per_serving_mg"] > 0) & (df["is_proprietary_blend"] == 0)]
for ing in ["Magnesium", "Vitamin C", "Zinc", "Creatine", "Melatonin"]:
    s = disclosed.loc[disclosed["ingredient"] == ing, "amount_per_serving_mg"]
    if len(s):
        print(f"{ing:12} n={len(s):3d}  median={s.median():>8.3f} mg  range={s.min():g}-{s.max():g} mg")

## Proprietary blends - undisclosed doses, flagged (not dropped)

A differentiator: ingredients whose per-serving dose is hidden on the label are still listed, flagged `is_proprietary_blend == 1` with `amount = 0`.

In [ ]:
blend = df[df["is_proprietary_blend"] == 1]
print(f"{len(blend):,} blend ingredient rows ({len(blend)/len(df):.0%} of the sample) across {blend['product_id'].nunique()} products")
# products with the most undisclosed-dose ingredients
blend.groupby(["brand", "product_name"]).size().sort_values(ascending=False).head(5)

## Chemistry + canonicalization (NIH PubChem)

Compounds carry `pubchem_cid`, `molecular_formula`, `molecular_weight`, `canonical_smiles`, and `inchikey`. The **InChIKey** is a canonical chemical key: rows sharing one are the same molecule under different label names.

In [ ]:
have_chem = df["pubchem_cid"].notna().mean()
print(f"Rows with PubChem chemistry: {have_chem:.0%}")

# Canonicalization: one InChIKey -> possibly several label names (same molecule)
canon = (df.dropna(subset=["inchikey"])
           .groupby("inchikey")["ingredient"].nunique()
           .sort_values(ascending=False))
print(f"InChIKeys mapping to >1 distinct ingredient name: {(canon > 1).sum()}")
example_key = canon[canon > 1].index[0]
print("\nExample - same molecule, different label names:")
print(f"  InChIKey {example_key}")
print(" ", sorted(df.loc[df["inchikey"] == example_key, "ingredient"].unique()))

## Provenance - every record is re-verifiable

`dsld_label_id` + `source_url` link each row back to its original NIH DSLD label.

In [ ]:
df[["brand", "product_name", "ingredient", "amount_per_serving_mg", "pubchem_cid", "source_url"]].head(6)

---
**Full dataset (17,000+ products, SQLite + CSV + JSON):** [suppdb.net](https://supplements-nootropics-suppdb.pages.dev) - questions: suppdb.doorframe589@simplelogin.com